# 01 · Logistic Regression Baseline

**Competition:** Kaggle Playground Series S6E9 — Predicting Electric Vehicle Purchases  
**Metric:** ROC-AUC

This notebook presents the project's initial Logistic Regression baseline in a compact, reproducible form.

The baseline is intentionally simple:

1. remove `id` and the target;
2. one-hot encode categorical variables;
3. standardize all encoded features;
4. evaluate with 5-fold stratified cross-validation;
5. refit on the full training set and generate the test submission.

Its role is to establish a transparent reference point before introducing tree models, richer feature representations, target encoding, and model ensembles.

> Reference result: OOF AUC was approximately **0.9381**.

## 1. Imports and paths

The notebook expects the competition files to be available under `DATA_DIR`:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

No package installation, path auto-discovery, checkpointing, or competition-session recovery logic is included here. Those were useful during experimentation but are not part of the statistical method.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

SEED = 42
N_FOLDS = 5
TARGET = "Will_Buy_EV"
ID_COL = "id"

DATA_DIR = Path(".")
OUTPUT_DIR = Path("outputs/01_logistic_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

## 2. Load data

The target is mapped from `No/Yes` to `0/1`. The competition identifier is excluded from the model because it is not treated as a predictive feature in this baseline.

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

X = train.drop(columns=[ID_COL, TARGET])
y = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")
X_test = test.drop(columns=[ID_COL])

summary = pd.Series({
    "train_rows": len(train),
    "test_rows": len(test),
    "raw_features": X.shape[1],
    "positive_rate": y.mean(),
})
summary

## 3. One-hot encoding

`pd.get_dummies` converts categorical variables into indicator columns. Train and test are then aligned so that the model receives exactly the same feature columns in both datasets.

This transformation does not use the target. The baseline deliberately keeps feature engineering minimal.

In [ ]:
X_encoded = pd.get_dummies(X, dtype=np.int8)
X_test_encoded = pd.get_dummies(X_test, dtype=np.int8)

X_encoded, X_test_encoded = X_encoded.align(
    X_test_encoded,
    join="left",
    axis=1,
    fill_value=0,
)

assert X_encoded.columns.equals(X_test_encoded.columns)

print("Encoded train shape:", X_encoded.shape)
print("Encoded test shape: ", X_test_encoded.shape)

## 4. Five-fold OOF validation

We use `StratifiedKFold` because the target is imbalanced and each fold should preserve approximately the same positive-class proportion.

The scaler is fitted **inside each training fold**:

$$
\mu_j
=
\frac{1}{n_{\text{train}}}
\sum_{i \in \text{train}} x_{ij},
\qquad
\sigma_j^2
=
\frac{1}{n_{\text{train}}}
\sum_{i \in \text{train}}(x_{ij}-\mu_j)^2
$$

and validation rows are transformed with those training-fold statistics.

This avoids using validation-fold distribution statistics when fitting the scaler.

The OOF vector contains one prediction for every training row, produced by a model that did not train on that row.

In [ ]:
cv = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED,
)

oof_pred = np.zeros(len(train), dtype=float)
fold_rows = []

for fold, (train_idx, valid_idx) in enumerate(
    cv.split(X_encoded, y),
    start=1,
):
    X_train = X_encoded.iloc[train_idx]
    X_valid = X_encoded.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_valid_scaled = scaler.transform(X_valid)

    model = LogisticRegression(
        max_iter=1000,
        random_state=SEED,
    )
    model.fit(X_train_scaled, y_train)

    valid_pred = model.predict_proba(X_valid_scaled)[:, 1]
    oof_pred[valid_idx] = valid_pred

    fold_auc = roc_auc_score(y_valid, valid_pred)
    fold_rows.append({
        "fold": fold,
        "auc": fold_auc,
        "train_rows": len(train_idx),
        "valid_rows": len(valid_idx),
    })

cv_results = pd.DataFrame(fold_rows)
oof_auc = roc_auc_score(y, oof_pred)

print(cv_results[["fold", "auc"]].to_string(index=False))
print()
print(f"Mean fold AUC: {cv_results['auc'].mean():.6f}")
print(f"Fold AUC std:  {cv_results['auc'].std(ddof=0):.6f}")
print(f"OOF AUC:       {oof_auc:.6f}")

### Why report pooled OOF AUC?

The mean of fold AUCs is useful for checking fold-to-fold stability, but the project uses the pooled OOF prediction vector as the main comparison object:

$$
\mathrm{AUC}_{\mathrm{OOF}}
=
\mathrm{AUC}(y,\hat p_{\mathrm{OOF}})
$$

This produces one directly comparable score for the entire training set and preserves the OOF predictions for later ensemble analysis.

In [ ]:
oof_output = pd.DataFrame({
    ID_COL: train[ID_COL],
    "y_true": y,
    "oof_pred": oof_pred,
})

oof_output.to_csv(
    OUTPUT_DIR / "oof_logistic.csv",
    index=False,
)
cv_results.to_csv(
    OUTPUT_DIR / "cv_metrics.csv",
    index=False,
)

## 5. Refit on the full training set

Cross-validation estimates generalization performance. After the baseline has been evaluated, the original competition workflow refits the same preprocessing and Logistic Regression model on all available training rows, then predicts the test set.

This matches the final-submission logic of the original baseline notebook.

In [ ]:
final_scaler = StandardScaler()
X_full_scaled = final_scaler.fit_transform(X_encoded)
X_test_scaled = final_scaler.transform(X_test_encoded)

final_model = LogisticRegression(
    max_iter=1000,
    random_state=SEED,
)
final_model.fit(X_full_scaled, y)

test_pred = final_model.predict_proba(X_test_scaled)[:, 1]

## 6. Generate the baseline submission

In [ ]:
submission = sample_submission.copy()
submission[TARGET] = test_pred

submission_path = OUTPUT_DIR / "submission_logistic.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
submission.head()

## 7. Baseline interpretation

This model is intentionally limited:

- it is linear in the encoded feature space;
- it does not learn tree-style threshold effects;
- it does not explicitly model higher-order interactions;
- it uses no target encoding, frequency encoding, income digit decomposition, or specialized feature views.

That simplicity is useful. The Logistic Regression OOF AUC provides a clean baseline against which later gains from tree models, feature representation, cross-fitted target statistics, and ensemble diversity can be measured.

The next stage of the project moves to a higher-resolution LightGBM feature view.

## Result

OOF AUC: **0.938099**

This baseline provides the reference point for the nonlinear tree models and feature-engineering experiments that follow.